# Master thesis separation visual examples

This is the separated thesis export workflow from `evaluate_pin_spot_separation_visual_examples.ipynb`. It uses the combined model-level CSVs directly and only joins spot-level metrics when a real spot-level CSV with `spot_channel` exists, avoiding `Export failed: 'spot_channel'`.

The model-output extraction path follows `eva_server/evaluation/export_selected_model_outputs.py`.

In [ ]:
from pathlib import Path

import h5py
try:
    import hdf5plugin  # noqa: F401 - registers compressed HDF5 filters
except ImportError:
    hdf5plugin = None
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import re
import pandas as pd
import torch
import torch.nn.functional as F

from IPython.display import display

from unet_model import UNet


## Configuration

Edit this cell for server-specific data, checkpoint, CSV, and export paths.

In [ ]:
# ==================== EDIT THIS CELL ====================
DATA_PATH = Path("data_100000_spots/augmented_spots_train.h5")
# For the extra evaluation run on a different dataset, switch DATA_PATH to that HDF5.
#DATA_PATH = Path("PATH/TO/OTHER_DATASET.h5")
OUTPUT_ROOT = Path("evaluation/master_thesis_visual_examples")
EXAMPLE_SPOTS_ROOT = OUTPUT_ROOT / "example_spots"

# Placeholder CSV paths: replace these with the new model-level CSV exports before running.
# Spot-level metrics are optional and discovered only when a companion CSV with a
# spot_channel column exists next to the model-level CSV.
COMBINED_CSV_FILES = [
    Path("PATH/TO/combined_multioutput_model_separation_metrics_5000.csv"),
    Path("PATH/TO/combined_oneoutput_model_separation_metrics_5000.csv"),
    Path("PATH/TO/combined_twooutput_model_separation_metrics_5000.csv"),
]

# Add external-server directories here if model_path entries in the CSV are relative.
# Example: Path("/scratch/your-user/masterarbeit/checkpoints")
CHECKPOINT_ROOTS = [
    Path("."),
    Path("checkpoints"),
]

MODEL_RENAMES = {
    # Current run-id naming scheme: final date-time plus output variant.
    "0709-1413_pin": "multi-output BCE/MSE loss",
    "0721-2040_pin": "multi-output Tversky loss batch 100",
    "0720-1538_pin": "multi-output Tversky loss base channel 32",
    "0704-0227_pin": "multi-output BCE/MSE loss small Dataset",
    "0729-0433_pin": "multi-output Tversky loss learning rate 3e-4",
    "0728-0732_pin": "multi-output Tversky loss learning rate 5e-5",
    "0802-1237_pin": "multi-output Tversky loss batch 20",
    "0803-1354_pin": "multi-output Tversky loss learning rate 1e-4",
    "0804-2249_pin": "multi-output Tversky loss ReduceLROnPlateau 0.5",
    "0702-1451_oneoutput": "one-output Dice/L1 loss",
    "0727-1529_oneoutput": "one-output Tversky loss",
    "0719-0847_twooutput": "two-output L1 loss batch 100",
    "0721-1027_twooutput": "two-output reconstruction loss",
    "0721-0111_twooutput": "two-output L1 loss",
    "0717-0844_twooutput": "two-output L1 loss base channel 32",
    "0707-1118_twooutput": "two-output Tversky loss",
    "0703-2013_twooutput": "two-output Tversky loss small Dataset",
    "0804-2249_unknown": "multi-output Tversky loss ReduceLROnPlateau 0.5 evaluation",

    # Legacy labels kept so older CSVs still normalize correctly.
    "one-output Loss": "one-output Dice/L1 loss",
    "one-output Tversky Loss": "one-output Tversky loss",
    ("multi" "-head BCE/MSE Loss"): "multi-output BCE/MSE loss",
    ("multi" "-head BCE/MSE loss"): "multi-output BCE/MSE loss",
    ("multi" "-head Tversky Loss batch 100"): "multi-output Tversky loss batch 100",
    ("multi" "-head Tversky loss batch 100"): "multi-output Tversky loss batch 100",
    ("multi" "-head Tversky Loss base-channel 32"): "multi-output Tversky loss base channel 32",
    ("multi" "-head Tversky loss base channel 32"): "multi-output Tversky loss base channel 32",
    ("multi" "-head BCE/MSE Loss small Dataset"): "multi-output BCE/MSE loss small Dataset",
    ("multi" "-head BCE/MSE loss small Dataset"): "multi-output BCE/MSE loss small Dataset",
    ("multi" "-head Tversky Loss lr 3e-04"): "multi-output Tversky loss learning rate 3e-4",
    ("multi" "-head Tversky loss learning rate 3e-4"): "multi-output Tversky loss learning rate 3e-4",
    ("multi" "-head Tversky Loss lr 5e-05"): "multi-output Tversky loss learning rate 5e-5",
    ("multi" "-head Tversky loss learning rate 5e-5"): "multi-output Tversky loss learning rate 5e-5",
    ("multi" "-head Tversky Loss batch 20"): "multi-output Tversky loss batch 20",
    ("multi" "-head Tversky loss batch 20"): "multi-output Tversky loss batch 20",
    ("multi" "-head Tversky Loss lr 1e-04"): "multi-output Tversky loss learning rate 1e-4",
    ("multi" "-head Tversky loss learning rate 1e-4"): "multi-output Tversky loss learning rate 1e-4",
    ("multi" "-head Tversky Loss ReduceLROnPlateau 0.5"): "multi-output Tversky loss ReduceLROnPlateau 0.5",
    ("multi" "-head Tversky loss ReduceLROnPlateau 0.5"): "multi-output Tversky loss ReduceLROnPlateau 0.5",
    ("multi" "-head Tversky Loss ReduceLROnPlateau 0.5 Evaluation"): "multi-output Tversky loss ReduceLROnPlateau 0.5 evaluation",
    ("multi" "-head Tversky loss ReduceLROnPlateau 0.5 evaluation"): "multi-output Tversky loss ReduceLROnPlateau 0.5 evaluation",
    "two-output L1 Loss batch 100": "two-output L1 loss batch 100",
    "two-output Reconstruction Loss": "two-output reconstruction loss",
    "two-output L1 Loss": "two-output L1 loss",
    "two-output L1 base-channel 32": "two-output L1 loss base channel 32",
    "two-output Tversky Loss": "two-output Tversky loss",
    "two-output Tversky Loss small Dataset": "two-output Tversky loss small Dataset",
}

EXCLUDED_MODELS = set()

# Empty THESIS_MODELS = no in-depth exports. Empty APPENDIX_MODELS = every loaded model when APPENDIX_SAMPLES is not empty.
THESIS_MODELS = []
APPENDIX_MODELS = []

# None = automatically use this model's best/worst sample.
# Otherwise use a numeric suffix ("123") or complete name ("sample_000123").
# Use "samples" for an ordered thesis-style export without good/bad grouping.
THESIS_SAMPLE_OVERRIDES = {}

# Shared samples for the appendix/example-spots export. Numeric suffixes are accepted.
APPENDIX_SAMPLES = [
    "sample_026386",
    "sample_022778",
    "sample_049479",
    "sample_063684",
    "sample_054747",
    "sample_078242",
]

THESIS_LAYOUT = "Thesis comparison (recommended)"
APPENDIX_LAYOUT = "Model output/errors only"
APPENDIX_SAVE_INDIVIDUAL_PDFS = False
APPENDIX_SAVE_MULTIPAGE_PDFS = False
APPENDIX_SAVE_COMBINED_BY_SPOT_PDFS = True
MODELS_PER_OUTPUT_PAGE = 6

PDF_DPI = 300
IMG_SCALE = 1.0
PREDICTION_MASK_THRESHOLD = 0.5
INTENSITY_MASK_THRESHOLD = 1e-4
BILINEAR = False  # must match checkpoint training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ERROR_FRAME_BACKGROUND = "white"  # choose "white" or "black"
STRICT_THESIS_COUNTS = False
THESIS_EXAMPLES_PER_ROLE = 5
BEST_WORST_SPOTS_PER_MODEL = 3
# =============================================================

try:
    font_manager.findfont("CMU Serif", fallback_to_default=False)
    FIGURE_FONT = "CMU Serif"
except ValueError:
    FIGURE_FONT = "Computer Modern Roman"
    print(
        "CMU Serif was not found in this environment. Install the CMU fonts and "
        "restart the kernel for exact CMU typography; using Computer Modern fallback."
    )

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": [FIGURE_FONT, "CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
        "mathtext.fontset": "cm",
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

print(f"Device: {DEVICE}")
print(f"Figure font: {FIGURE_FONT}")


In [ ]:
def dataset_label(experiment):
    text = str(experiment).replace(".h5", "").replace("_logtif", "")
    if text in {"Al_segvol", "Al"}:
        return "Al"
    if text.startswith("Al_deformed_LoG"):
        return "Al_deformed_LoG"
    if text.startswith("Al_big_grains"):
        return "Al_big_grains"
    if text.startswith("Al_small_grains"):
        return "Al_small_grains"
    if text.startswith("Cu"):
        return "Cu"
    if text.startswith("IN718_twins"):
        return "IN718_twins"
    if text.startswith("Iron_deformed"):
        return "Iron_deformed"
    if text.startswith("Iron"):
        return "Iron"
    if text.startswith("Ti7Al"):
        return "Ti7Al"
    return text


def infer_architecture(row):
    if "architecture" in row and pd.notna(row["architecture"]):
        return str(row["architecture"])
    text = " ".join(str(row.get(col, "")) for col in ["source_file", "source_results_csv", "model_path", "model"])
    if "oneoutput" in text:
        return "oneoutput"
    if "twooutput" in text or "two-output" in text:
        return "twooutput"
    return "pin"


In [ ]:
def normalize_image_and_targets(image, targets):
    image = image.astype(np.float32, copy=False)
    targets = targets.astype(np.float32, copy=False)
    finite = np.isfinite(image)
    if not finite.any():
        return np.zeros_like(image, dtype=np.float32), np.zeros_like(targets, dtype=np.float32)

    values = image[finite]
    lo, hi = np.percentile(values, [1, 99.9])
    if hi <= lo:
        lo, hi = float(values.min()), float(values.max())
    if hi <= lo:
        return np.zeros_like(image, dtype=np.float32), np.zeros_like(targets, dtype=np.float32)

    image = np.clip(image, lo, hi)
    image = (image - lo) / (hi - lo)
    image[~finite] = 0.0
    targets = np.clip(targets, lo, hi)
    targets = (targets - lo) / (hi - lo)
    targets[~np.isfinite(targets)] = 0.0
    return image.astype(np.float32), targets.astype(np.float32)


def load_sample(sample_name):
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"HDF5 data not found at {DATA_PATH.resolve()}. Update DATA_PATH in the configuration."
        )
    with h5py.File(DATA_PATH, "r") as h5_file:
        if sample_name not in h5_file:
            raise KeyError(f"{sample_name!r} is not present in {DATA_PATH}.")
        group = h5_file[sample_name]
        image = group["image"][()]
        target_intensity = group["spot_images"][()]
        target_mask = group["spot_masks"][()] if "spot_masks" in group else target_intensity > INTENSITY_MASK_THRESHOLD

    if image.ndim == 3:
        image = image.mean(axis=-1)
    if target_intensity.shape[0] != 2 or target_mask.shape[0] != 2:
        raise ValueError(
            f"{sample_name}: expected two intensity and two mask channels; "
            f"got {target_intensity.shape} and {target_mask.shape}."
        )

    image, target_intensity = normalize_image_and_targets(image, target_intensity)
    target_mask = target_mask > 0
    image_tensor = torch.from_numpy(image).unsqueeze(0)
    intensity_tensor = torch.from_numpy(target_intensity)
    mask_tensor = torch.from_numpy(target_mask)

    if IMG_SCALE != 1.0:
        size = (
            max(1, int(image_tensor.shape[1] * IMG_SCALE)),
            max(1, int(image_tensor.shape[2] * IMG_SCALE)),
        )
        image_tensor = F.interpolate(
            image_tensor.unsqueeze(0), size=size, mode="bilinear", align_corners=False
        ).squeeze(0)
        intensity_tensor = F.interpolate(
            intensity_tensor.unsqueeze(0), size=size, mode="bilinear", align_corners=False
        ).squeeze(0)
        mask_tensor = F.interpolate(
            mask_tensor.float().unsqueeze(0), size=size, mode="nearest"
        ).squeeze(0).bool()
    return image_tensor, intensity_tensor, mask_tensor


def architecture_from_row(row):
    return infer_architecture(row)


def n_classes_for_architecture(architecture):
    return {"pin": 4, "oneoutput": 1, "twooutput": 2}.get(str(architecture), 4)


def infer_base_features(state_dict):
    for key, value in state_dict.items():
        if key.endswith("inc.double_conv.0.weight") and getattr(value, "ndim", 0) == 4:
            return int(value.shape[0])
    return 32


def resolve_checkpoint(model_path, roots=None):
    roots = list(CHECKPOINT_ROOTS if roots is None else roots)
    original = Path(str(model_path))
    candidates = []
    if original.is_absolute():
        candidates.append(original)
    else:
        candidates.append(original)
        for root in roots:
            root = Path(root)
            candidates.extend([root / original, root / original.name, root / "checkpoints" / original.name])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def load_model(model_path, architecture):
    checkpoint_path = resolve_checkpoint(model_path)
    if checkpoint_path is None:
        raise FileNotFoundError(
            f"Checkpoint not found for {model_path!r}. Add the server checkpoint directory to CHECKPOINT_ROOTS."
        )
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict):
        checkpoint.pop("mask_values", None)
    model = UNet(
        n_channels=1,
        n_classes=n_classes_for_architecture(architecture),
        bilinear=BILINEAR,
        base_features=infer_base_features(checkpoint) if isinstance(checkpoint, dict) else 32,
    ).to(DEVICE)
    model.load_state_dict(checkpoint)
    model.eval()
    return model


_model_cache = {"key": None, "model": None}


def cached_model(model_path, architecture):
    key = (str(model_path), str(architecture))
    if _model_cache["key"] != key:
        _model_cache["model"] = None
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        _model_cache["model"] = load_model(model_path, architecture)
        _model_cache["key"] = key
    return _model_cache["model"]


def apply_target_assignment(row, target_intensity, target_mask):
    if str(row.get("selected_target_for_output", "target_0")) == "target_1":
        return target_intensity.flip(0), target_mask.flip(0)
    return target_intensity, target_mask


def apply_output_assignment(row, pred_intensity, pred_mask_probability=None):
    if str(row.get("channel_assignment", "direct")) == "swapped":
        pred_intensity = pred_intensity.flip(1)
        if pred_mask_probability is not None:
            pred_mask_probability = pred_mask_probability.flip(1)
    return pred_intensity, pred_mask_probability


def predict_sample(row):
    architecture = architecture_from_row(row)
    image, target_intensity, target_mask = load_sample(row["sample_name"])
    model = cached_model(row["model_path"], architecture)
    image_batch = image.unsqueeze(0).to(DEVICE, dtype=torch.float32)

    with torch.no_grad():
        logits = model(image_batch)
        if logits.shape[2:] != target_intensity.shape[1:]:
            logits = F.interpolate(
                logits,
                size=target_intensity.shape[1:],
                mode="bilinear",
                align_corners=False,
            )

        if architecture == "oneoutput":
            first_spot = torch.sigmoid(logits) * image_batch
            pred_intensity = torch.cat([first_spot, (image_batch - first_spot).clamp_min(0.0)], dim=1)
            pred_mask_probability = None
            target_intensity, target_mask = apply_target_assignment(row, target_intensity, target_mask)
        elif architecture == "twooutput":
            pred_intensity = torch.sigmoid(logits)
            pred_mask_probability = None
            pred_intensity, _ = apply_output_assignment(row, pred_intensity)
        else:
            pred_mask_probability = torch.sigmoid(logits[:, 0:2])
            pred_intensity = torch.sigmoid(logits[:, 2:4])
            pred_intensity, pred_mask_probability = apply_output_assignment(row, pred_intensity, pred_mask_probability)
    pred_intensity_np = pred_intensity[0].cpu().numpy()
    pred_mask_probability_np = None if pred_mask_probability is None else pred_mask_probability[0].cpu().numpy()
    pred_mask_np = None
    if pred_mask_probability_np is not None:
        pred_mask_np = pred_mask_probability_np > PREDICTION_MASK_THRESHOLD

    return {
        "architecture": architecture,
        "image": image[0].numpy(),
        "target_mask": target_mask.numpy(),
        "target_intensity": target_intensity.numpy(),
        "pred_mask_probability": pred_mask_probability_np,
        "pred_mask": pred_mask_np,
        "pred_intensity": pred_intensity_np,
    }


In [ ]:
MASK_COLORS = ["#35b779", "#3b4cc0"]
ERROR_MISSING_COLOR = np.array([0.54, 0.17, 0.89], dtype=np.float32)  # purple: target missing in prediction
ERROR_EXTRA_COLOR = np.array([0.10, 0.70, 0.36], dtype=np.float32)  # green: prediction has too much


def finish_axes(axes):
    for ax in np.asarray(axes).flat:
        ax.set_xticks([])
        ax.set_yticks([])


def panel_text(row, channel):
    parts = []
    for col, label in [("dice", "Dice"), ("rmse", "RMSE")]:
        value = row.get(f"spot_{channel}_{col}")
        if pd.notna(value) if value is not None else False:
            parts.append(f"{label}={value:.4f}")
    return "\n".join(parts)


def add_metric_label(ax, text):
    if text:
        ax.text(
            0.02,
            0.98,
            text,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            color="black" if ERROR_FRAME_BACKGROUND == "white" else "white",
            bbox={
                "facecolor": "white" if ERROR_FRAME_BACKGROUND == "white" else "black",
                "alpha": 0.72,
                "pad": 2,
                "edgecolor": "none",
            },
        )


def signed_error_rgb(missing, extra):
    missing = np.clip(np.asarray(missing, dtype=np.float32), 0.0, None)
    extra = np.clip(np.asarray(extra, dtype=np.float32), 0.0, None)
    scale = max(float(np.nanmax(missing)) if missing.size else 0.0, float(np.nanmax(extra)) if extra.size else 0.0, 1e-8)
    missing = np.clip(missing / scale, 0.0, 1.0)
    extra = np.clip(extra / scale, 0.0, 1.0)
    base = np.ones((*missing.shape, 3), dtype=np.float32) if ERROR_FRAME_BACKGROUND == "white" else np.zeros((*missing.shape, 3), dtype=np.float32)
    color = base * (1.0 - np.maximum(missing, extra))[..., None]
    color += missing[..., None] * ERROR_MISSING_COLOR
    color += extra[..., None] * ERROR_EXTRA_COLOR
    return np.clip(color, 0.0, 1.0)


def error_panel(data, channel):
    if data["architecture"] == "pin" and data.get("pred_mask") is not None:
        target = data["target_mask"][channel].astype(bool)
        pred = data["pred_mask"][channel].astype(bool)
        return signed_error_rgb(target & ~pred, pred & ~target)
    target = np.asarray(data["target_intensity"][channel], dtype=np.float32)
    pred = np.asarray(data["pred_intensity"][channel], dtype=np.float32)
    return signed_error_rgb(target - pred, pred - target)


def show_panel(ax, panel, cmap="gray", vmin=0, vmax=1, is_error=False):
    if is_error:
        ax.set_facecolor(ERROR_FRAME_BACKGROUND)
        return ax.imshow(panel)
    return ax.imshow(panel, cmap=cmap, vmin=vmin, vmax=vmax)


def intensity_error(data, channel):
    return error_panel(data, channel)


def combined_predicted_mask_panel(data):
    masks = data.get("pred_mask_probability")
    if masks is None:
        return data["image"]
    masks = np.clip(np.asarray(masks, dtype=np.float32), 0.0, 1.0)
    panel = np.zeros((*masks.shape[1:], 3), dtype=np.float32)
    colors = np.array([mpl.colors.to_rgb(color) for color in MASK_COLORS], dtype=np.float32)
    panel += masks[0][..., None] * colors[0]
    panel += masks[1][..., None] * colors[1]
    return np.clip(panel, 0.0, 1.0)


def plot_ground_truth_single(data, title, row=None):
    fig, axes = plt.subplots(1, 5, figsize=(14.5, 3.1), constrained_layout=True)
    panels = [
        (data["image"], "input", "gray"),
        (data["target_mask"][0], "mask 1", "gray"),
        (data["target_mask"][1], "mask 2", "gray"),
        (data["target_intensity"][0], "intensity 1", "gray"),
        (data["target_intensity"][1], "intensity 2", "gray"),
    ]
    for ax, (panel, panel_title, cmap) in zip(axes, panels):
        show_panel(ax, panel, cmap=cmap)
        ax.set_title(panel_title)
    finish_axes(axes)
    fig.suptitle(title)
    return fig


def plot_model_outputs_only(data, title, row=None):
    fig, axes = plt.subplots(2, 2, figsize=(6.2, 6.0), constrained_layout=True)
    for channel in range(2):
        panels = [
            (data["pred_intensity"][channel], f"output {channel + 1}", "gray", False),
            (error_panel(data, channel), f"error {channel + 1}", None, True),
        ]
        for column, (panel, panel_title, cmap, is_error) in enumerate(panels):
            show_panel(axes[channel, column], panel, cmap=cmap or "gray", is_error=is_error)
            axes[channel, column].set_title(panel_title)
            if row is not None and is_error:
                add_metric_label(axes[channel, column], panel_text(row, channel))
    finish_axes(axes)
    fig.suptitle(title)
    return fig


def plot_thesis_comparison(data, title, row=None):
    show_masks = data["architecture"] == "pin" and data.get("pred_mask") is not None
    ncols = 4
    fig, axes = plt.subplots(2, ncols, figsize=(3.0 * ncols, 6.0), constrained_layout=True)
    for channel in range(2):
        first_panel = data["image"]
        first_title = "input" if channel == 0 else "input 2"
        first_cmap = "gray"
        if show_masks and channel == 1:
            first_panel = combined_predicted_mask_panel(data)
            first_title = "predicted masks"
            first_cmap = None
        panels = [
            (first_panel, first_title, first_cmap, False),
            (data["target_intensity"][channel], f"target {channel + 1}", "gray", False),
            (data["pred_intensity"][channel], f"prediction {channel + 1}", "gray", False),
            (error_panel(data, channel), f"error {channel + 1}", None, True),
        ]
        for column, (panel, panel_title, cmap, is_error) in enumerate(panels):
            show_panel(axes[channel, column], panel, cmap=cmap or "gray", is_error=is_error)
            axes[channel, column].set_title(panel_title)
            if column == 0 and show_masks and channel == 0:
                for mask_channel in range(2):
                    axes[channel, column].contour(
                        data["target_mask"][mask_channel],
                        levels=[0.5],
                        colors=[MASK_COLORS[mask_channel]],
                        linewidths=1.1,
                    )
            if row is not None and is_error:
                add_metric_label(axes[channel, column], panel_text(row, channel))
    finish_axes(axes)
    fig.suptitle(title)
    return fig


def plot_full_comparison(data, title, row=None):
    return plot_thesis_comparison(data, title, row=row)


def plot_predictions_only(data, title, row=None):
    fig, axes = plt.subplots(2, 3, figsize=(9, 6), constrained_layout=True)
    for channel in range(2):
        panels = [
            (data["image"], f"input {channel + 1}", "gray", False),
            (data["pred_intensity"][channel], f"predicted intensity {channel + 1}", "gray", False),
            (error_panel(data, channel), f"error {channel + 1}", None, True),
        ]
        for column, (panel, panel_title, cmap, is_error) in enumerate(panels):
            show_panel(axes[channel, column], panel, cmap=cmap or "gray", is_error=is_error)
            axes[channel, column].set_title(panel_title)
    finish_axes(axes)
    fig.suptitle(title)
    return fig


def plot_overlay_errors(data, title, row=None):
    return plot_thesis_comparison(data, title, row=row)


def plot_ground_truth(data, title, row=None):
    return plot_ground_truth_single(data, title, row=row)


LAYOUT_FUNCTIONS = {
    "Thesis comparison (recommended)": plot_thesis_comparison,
    "Thesis output": plot_thesis_comparison,
    "Predictions only": plot_predictions_only,
    "Overlay and errors": plot_overlay_errors,
    "Ground truth only": plot_ground_truth,
    "Ground truth single": plot_ground_truth_single,
    "Model output/errors only": plot_model_outputs_only,
}


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages


def normalize_model_name(name):
    return MODEL_RENAMES.get(str(name), str(name))


def normalize_model_list(names):
    return [normalize_model_name(name) for name in names]


def normalized_thesis_sample_overrides():
    return {normalize_model_name(model): overrides for model, overrides in THESIS_SAMPLE_OVERRIDES.items()}


def load_thesis_results():
    paths = [Path(x) for x in COMBINED_CSV_FILES]
    paths = list(dict.fromkeys(path.resolve() for path in paths))
    missing = [path for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing CSVs:\n" + "\n".join(map(str, missing)))
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame["source_results_csv"] = str(path)
        frames.append(frame)
    table = pd.concat(frames, ignore_index=True)
    required = {
        "model", "model_path", "sample_name",
        "spot_recognition_mean", "dice_mean", "iou_mean", "rmse_mean",
    }
    missing_columns = required.difference(table.columns)
    if missing_columns:
        raise ValueError(f"CSV files are missi01:ng columns:base chann {sorted(missing_columns)}")
    if "model_original" not in table:
        table["model_original"] = table["model"].astype(str)
    else:
        table["model_original"] = table["model_original"].fillna(table["model"]).astype(str)
    table["model"] = table["model"].astype(str).replace(MODEL_RENAMES)
    table["model"] = table["model_original"].replace(MODEL_RENAMES).where(
        table["model_original"].isin(MODEL_RENAMES), table["model"]
    )
    if "dataset" not in table:
        if "experiment" not in table:
            raise ValueError("CSV files need a dataset or experiment column.")
        table["dataset"] = table["experiment"].map(dataset_label)
    if "architecture" not in table:
        table["architecture"] = table.apply(infer_architecture, axis=1)
    if "channel_assignment" not in table:
        table["channel_assignment"] = "direct"
    if "selected_target_for_output" not in table:
        table["selected_target_for_output"] = "target_0"
    excluded = set(EXCLUDED_MODELS)
    if excluded:
        before = len(table)
        table = table.loc[
            ~table["model"].isin(excluded)
            & ~table["model_original"].isin(excluded)
        ].copy()
        removed = before - len(table)
        if removed:
            print(f"Dropped {removed:,} rows from excluded models.")
    appendix_wants_all_models = bool(APPENDIX_SAMPLES) and not APPENDIX_MODELS
    requested_models = set() if appendix_wants_all_models else (set(normalize_model_list(THESIS_MODELS)) | set(normalize_model_list(APPENDIX_MODELS)))
    if requested_models:
        known = set(table["model"].dropna().unique())
        missing = sorted(requested_models.difference(known))
        if missing:
            print("Warning: requested models not found:", ", ".join(missing))
        table = table.loc[table["model"].isin(requested_models)].copy()
    collisions = table.groupby("model")["model_path"].nunique()
    if (collisions > 1).any():
        names = collisions[collisions > 1].index.tolist()
        raise ValueError(f"Model-name collisions {names}; resolve them with MODEL_RENAMES.")
    return table, paths



def spot_csv_candidates_from_sample_csv(path):
    path = Path(path)
    name = path.name
    replacements = [
        ("combined_pin_model_separation_metrics", "combined_pin_spot_separation_metrics"),
        ("combined_oneoutput_model_separation_metrics", "combined_oneoutput_spot_separation_metrics"),
        ("combined_twooutput_model_separation_metrics", "combined_twooutput_spot_separation_metrics"),
        ("pin_model_separation_metrics", "pin_spot_separation_metrics"),
        ("oneoutput_model_separation_metrics", "oneoutput_spot_separation_metrics"),
        ("model_separation_metrics", "spot_separation_metrics"),
    ]
    candidates = []
    for old, new in replacements:
        if old in name:
            candidates.append(path.with_name(name.replace(old, new)))
    if path.parent.exists():
        candidates.extend(sorted(path.parent.glob("*spot_separation_metrics*.csv")))
    return list(dict.fromkeys(candidates))


def spot_csv_from_sample_csv(path):
    for candidate in spot_csv_candidates_from_sample_csv(path):
        if Path(candidate).exists():
            return Path(candidate)
    return None


def spot_csvs_for_row(row):
    sources = []
    for column in ["source_file", "source_results_csv"]:
        value = row.get(column)
        if value is not None and pd.notna(value):
            sources.append(value)
    candidates = []
    for source in sources:
        candidates.extend(spot_csv_candidates_from_sample_csv(source))
    return [Path(path) for path in dict.fromkeys(candidates) if Path(path).exists()]


def load_spot_rows_for_source(row):
    frames = []
    for spot_csv in spot_csvs_for_row(row):
        spot_rows = pd.read_csv(spot_csv)
        if "spot_channel" not in spot_rows.columns:
            print(f"Skipping {spot_csv}: no spot_channel column; this is not a spot-level metrics CSV.")
            continue
        mask = (
            (spot_rows["model"].astype(str) == str(row.get("model_original", row["model"])))
            & (spot_rows["model_path"].astype(str) == str(row["model_path"]))
            & (spot_rows["sample_name"].astype(str) == str(row["sample_name"]))
        )
        out = spot_rows.loc[mask].copy()
        if out.empty:
            continue
        out["model_display"] = row["model"]
        out["architecture"] = row.get("architecture", infer_architecture(row))
        out["dataset"] = row.get("dataset")
        out["source_spot_csv"] = str(spot_csv)
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def attach_spot_metrics(row):
    out = row.copy()
    spot_rows = load_spot_rows_for_source(row)
    if spot_rows.empty or "spot_channel" not in spot_rows.columns:
        return out
    for channel in range(2):
        match = spot_rows.loc[pd.to_numeric(spot_rows["spot_channel"], errors="coerce") == channel]
        if not match.empty:
            match = match.iloc[0]
            for metric in ["spot_recognition", "dice", "iou", "rmse", "nrmse", "integrated_intensity_abs_relative_error"]:
                if metric in match:
                    out[f"spot_{channel}_{metric}"] = match[metric]
    return out


def selected_spot_metrics(rows):
    frames = [load_spot_rows_for_source(row) for _, row in rows.iterrows()]
    frames = [frame for frame in frames if not frame.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def best_worst_spots_by_model(n=BEST_WORST_SPOTS_PER_MODEL):
    source_rows = thesis_results.drop_duplicates(["model", "model_original", "model_path", "source_file" if "source_file" in thesis_results else "source_results_csv"])
    frames = []
    for _, row in source_rows.iterrows():
        spot_frames = []
        for spot_csv in spot_csvs_for_row(row):
            spots = pd.read_csv(spot_csv)
            if "spot_channel" not in spots.columns:
                print(f"Skipping {spot_csv}: no spot_channel column; this is not a spot-level metrics CSV.")
                continue
            spots = spots.loc[
                (spots["model"].astype(str) == str(row.get("model_original", row["model"])))
                & (spots["model_path"].astype(str) == str(row["model_path"]))
            ].copy()
            if spots.empty:
                continue
            spots["model_display"] = row["model"]
            spots["architecture"] = row.get("architecture", infer_architecture(row))
            spots["source_spot_csv"] = str(spot_csv)
            spot_frames.append(spots)
        if not spot_frames:
            continue
        spots = pd.concat(spot_frames, ignore_index=True)
        best = spots.sort_values(["spot_recognition", "sample_name", "spot_channel"], ascending=[False, True, True]).head(n).copy()
        worst = spots.sort_values(["spot_recognition", "sample_name", "spot_channel"], ascending=[True, True, True]).head(n).copy()
        best["quality_group"] = "best"
        worst["quality_group"] = "worst"
        frames.extend([best, worst])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


thesis_results, thesis_csv_paths = load_thesis_results()
thesis_available_models = (
    thesis_results[["model", "architecture", "model_path", "source_results_csv"]]
    .drop_duplicates().sort_values(["architecture", "model"]).reset_index(drop=True)
)
print(f"Loaded {len(thesis_results):,} rows for {thesis_results['model'].nunique()} thesis model(s).")
display(thesis_available_models)

best_worst_spot_table = best_worst_spots_by_model()
if not best_worst_spot_table.empty:
    print(f"Loaded {BEST_WORST_SPOTS_PER_MODEL} best and {BEST_WORST_SPOTS_PER_MODEL} worst spot rows per model where spot CSVs are available.")
    display(best_worst_spot_table[[
        "model_display", "architecture", "quality_group", "sample_name", "spot_channel",
        "spot_recognition", "dice", "rmse", "source_spot_csv"
    ]])
else:
    print("No spot-level CSVs were found. The visual export can still run; spot-specific metric labels and best/worst spot tables will be omitted.")


In [ ]:
def normalize_sample_query(query):
    text = str(query).strip()
    if text.isdigit():
        return f"sample_{int(text):06d}"
    return text


def thesis_sample_row(query, model_name):
    rows = thesis_results.loc[thesis_results["model"] == model_name].copy()
    text = str(query).strip()
    exact = rows.loc[rows["sample_name"].astype(str) == text]
    if not exact.empty:
        matches = exact
    elif text.isdigit():
        suffix = pd.to_numeric(rows["sample_name"].astype(str).str.extract(r"(\d+)$", expand=False), errors="coerce")
        matches = rows.loc[suffix == int(text)]
    else:
        matches = rows.iloc[0:0]
    matches = matches.drop_duplicates("sample_name")
    if len(matches) == 1:
        return attach_spot_metrics(matches.iloc[0])
    if rows.empty:
        raise ValueError(f"No metadata rows for model {model_name!r}.")

    fallback = rows.sort_values(["spot_recognition_mean", "sample_name"], ascending=[False, True]).iloc[0].copy()
    fallback["sample_name"] = normalize_sample_query(query)
    fallback["spot_recognition_mean"] = np.nan
    fallback["dice_mean"] = np.nan
    fallback["iou_mean"] = np.nan
    fallback["rmse_mean"] = np.nan
    fallback["sample_metrics_available"] = False
    return fallback


def automatic_thesis_rows(model_name, role, count=THESIS_EXAMPLES_PER_ROLE):
    rows = thesis_results.loc[thesis_results["model"] == model_name].sort_values(
        ["spot_recognition_mean", "sample_name"], ascending=[role == "bad", True]
    )
    if rows.empty:
        raise ValueError(f"No samples for model {model_name!r}.")
    selected = rows.head(count).copy()
    return [attach_spot_metrics(row) for _, row in selected.iterrows()]


def automatic_thesis_row(model_name, role):
    return automatic_thesis_rows(model_name, role, count=1)[0]


def thesis_model_names():
    return normalize_model_list(THESIS_MODELS)

def validate_thesis_configuration():
    available = set(thesis_results["model"])
    thesis_models = thesis_model_names()
    appendix_models = normalize_model_list(APPENDIX_MODELS) if APPENDIX_MODELS else sorted(available)
    unknown = (set(thesis_models) | set(appendix_models)) - available
    if unknown:
        available_preview = sorted(available)
        raise ValueError(f"Unknown model names: {sorted(unknown)}\nAvailable model names after MODEL_RENAMES: {available_preview}")
    if THESIS_EXAMPLES_PER_ROLE < 1:
        raise ValueError("THESIS_EXAMPLES_PER_ROLE must be at least 1.")
    if STRICT_THESIS_COUNTS and len(APPENDIX_SAMPLES) != 3:
        raise ValueError(f"Configure exactly 3 APPENDIX_SAMPLES; got {len(APPENDIX_SAMPLES)}.")
    if not APPENDIX_SAMPLES and not thesis_models:
        raise ValueError("Configure APPENDIX_SAMPLES or make sure thesis CSVs contain models before exporting.")
    if len(set(thesis_models)) != len(thesis_models):
        raise ValueError("THESIS_MODELS contains duplicates.")
    if len(set(map(str, APPENDIX_SAMPLES))) != len(APPENDIX_SAMPLES):
        raise ValueError("APPENDIX_SAMPLES contains duplicates.")
    for layout in (THESIS_LAYOUT, APPENDIX_LAYOUT):
        if layout not in LAYOUT_FUNCTIONS:
            raise ValueError(f"Unknown layout {layout!r}.")
    return thesis_models, appendix_models


def build_thesis_plans():
    thesis_models, appendix_models = validate_thesis_configuration()
    deep = []
    overrides_by_model = normalized_thesis_sample_overrides()
    for model in thesis_models:
        overrides = overrides_by_model.get(model, {})
        if "samples" in overrides:
            requested_values = overrides["samples"]
            for rank, value in enumerate(requested_values, start=1):
                row = thesis_sample_row(value, model)
                deep.append({
                    "section": "in_depth",
                    "role": f"sample_{rank:02d}",
                    "role_group": "manual",
                    "rank_within_role": rank,
                    "selection": "manual",
                    "requested_sample": str(value),
                    "layout": THESIS_LAYOUT,
                    **row.to_dict(),
                })
            continue
        for role in ("good", "bad"):
            requested = overrides.get(role)
            if requested is None:
                rows = automatic_thesis_rows(model, role, THESIS_EXAMPLES_PER_ROLE)
                selection = "automatic"
            else:
                requested_values = requested if isinstance(requested, (list, tuple, set)) else [requested]
                rows = [thesis_sample_row(value, model) for value in requested_values]
                selection = "manual"
            for rank, row in enumerate(rows, start=1):
                deep.append({
                    "section": "in_depth",
                    "role": f"{role}_{rank:02d}",
                    "role_group": role,
                    "rank_within_role": rank,
                    "selection": selection,
                    "layout": THESIS_LAYOUT,
                    **row.to_dict(),
                })
    appendix = []
    for query in APPENDIX_SAMPLES:
        for model in appendix_models:
            row = thesis_sample_row(query, model)
            appendix.append({"section":"appendix", "role":"shared sample", "selection":"manual", "requested_sample":str(query), "layout":APPENDIX_LAYOUT, **row.to_dict()})
    return pd.DataFrame(deep), pd.DataFrame(appendix)


PLAN_COLUMNS = [
    "section", "role", "role_group", "rank_within_role", "selection", "model", "architecture", "sample_name", "dataset",
    "spot_recognition_mean", "dice_mean", "rmse_mean",
    "spot_0_dice", "spot_0_rmse", "spot_1_dice", "spot_1_rmse",
    "layout", "model_path",
]


def existing_columns(df, columns):
    return [column for column in columns if column in df.columns]


def preview_thesis_plans():
    deep, appendix = build_thesis_plans()
    print("In-depth plan")
    display(deep[existing_columns(deep, PLAN_COLUMNS)])
    print("Shared-sample appendix plan")
    display(appendix[existing_columns(appendix, PLAN_COLUMNS)])
    return deep, appendix


def preview_thesis_figures(deep=None, appendix=None, show_deep=True, show_appendix=True):
    if deep is None or appendix is None:
        deep, appendix = build_thesis_plans()
    shown = 0
    if show_deep and not deep.empty:
        print("In-depth preview")
        fig = plan_figure(deep.iloc[0])
        display(fig)
        plt.close(fig)
        shown += 1
    if show_appendix and not appendix.empty:
        print("Appendix dense-page preview")
        preview_appendix_dense_page(appendix)
        shown += 1
    if shown == 0:
        print("No figures selected for preview.")
    return deep, appendix


## Preview export plan

Run this before exporting to check models, samples, metrics, and paths.

In [ ]:
deep_plan, appendix_plan = preview_thesis_plans()


In [ ]:
def format_metric(row, column):
    value = row.get(column)
    return "n/a" if value is None or pd.isna(value) else f"{value:.4f}"


def row_metrics_title(row):
    return (
        f"recognition={format_metric(row, 'spot_recognition_mean')}, "
        f"Dice={format_metric(row, 'dice_mean')}, "
        f"IoU={format_metric(row, 'iou_mean')}, "
        f"RMSE={format_metric(row, 'rmse_mean')}"
    )


def figure_title(row):
    layout = str(row.get("layout", ""))
    if layout.startswith("Thesis"):
        return f"{row['model']} | {row['sample_name']} "
    return f"{row['model']} | {row['dataset']} | {row['sample_name']} | {row['role']}"


def plan_figure(row):
    return LAYOUT_FUNCTIONS[row["layout"]](predict_sample(row), figure_title(row), row=row)


def safe_filename_part(value):
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "-", str(value)).strip("-_.")
    return cleaned or "unknown"


def save_plan_pdf(row, folder):
    layout = str(row["layout"]).replace(" (recommended)", "")
    architecture = row.get("architecture", infer_architecture(row))
    path = folder / ("__".join([
        safe_filename_part(architecture), safe_filename_part(row["model"]), safe_filename_part(row["role"]).lower(),
        safe_filename_part(row["sample_name"]), safe_filename_part(layout).lower(),
    ]) + ".pdf")
    fig = plan_figure(row)
    fig.savefig(path, format="pdf", dpi=PDF_DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def save_appendix_ground_truth_pdf(sample_name, row, folder):
    data = predict_sample(row)
    title = f"Ground truth | {row['dataset']} | {sample_name}"
    fig = plot_ground_truth_single(data, title, row=row)
    path = folder / f"appendix__{safe_filename_part(sample_name)}__ground-truth.pdf"
    fig.savefig(path, format="pdf", dpi=PDF_DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def output_page_title(sample_name, dataset):
    return f"{dataset} | {sample_name} | appendix comparison"


def mask_error_panel(data, channel):
    target = data["target_mask"][channel].astype(bool)
    pred = data.get("pred_mask")
    if pred is None:
        return None
    pred_channel = pred[channel].astype(bool)
    return signed_error_rgb(target & ~pred_channel, pred_channel & ~target)


def show_appendix_panel(ax, panel, title, cmap="gray", is_error=False):
    show_panel(ax, panel, cmap=cmap, is_error=is_error)
    ax.set_title(title, fontsize=7.5, pad=1.5)


def appendix_model_rows(data):
    rows = []
    if data["architecture"] == "pin" and data.get("pred_mask") is not None:
        rows.extend([
            ("mask", [
                (data["pred_mask_probability"][0], "mask 1", "gray", False),
                (data["pred_mask_probability"][1], "mask 2", "gray", False),
            ]),
            ("mask error", [
                (mask_error_panel(data, 0), "mask error 1", None, True),
                (mask_error_panel(data, 1), "mask error 2", None, True),
            ]),
        ])
    rows.extend([
        ("prediction", [
            (data["pred_intensity"][0], "pred 1", "gray", False),
            (data["pred_intensity"][1], "pred 2", "gray", False),
        ]),
        ("error", [
            (intensity_error(data, 0), "error 1", None, True),
            (intensity_error(data, 1), "error 2", None, True),
        ]),
    ])
    return rows


def draw_ground_truth(subfig, data, sample_name, dataset):
    subfig.suptitle(f"Ground truth | {dataset} | {sample_name}", fontsize=8.5)
    axes = subfig.subplots(1, 5)
    panels = [
        (data["image"], "input", "gray", False),
        (data["target_mask"][0], "mask 1", "gray", False),
        (data["target_mask"][1], "mask 2", "gray", False),
        (data["target_intensity"][0], "intensity 1", "gray", False),
        (data["target_intensity"][1], "intensity 2", "gray", False),
    ]
    for ax, (panel, title, cmap, is_error) in zip(np.asarray(axes).flat, panels):
        show_appendix_panel(ax, panel, title, cmap=cmap, is_error=is_error)
    finish_axes(axes)


def draw_appendix_model_block(subfig, row, data, max_rows):
    subfig.suptitle(str(row["model"]), fontsize=8.5)
    axes = subfig.subplots(max_rows, 2, squeeze=False)
    rows = appendix_model_rows(data)
    for row_index, (_, panels) in enumerate(rows):
        for col, (panel, title, cmap, is_error) in enumerate(panels):
            ax = axes[row_index, col]
            show_appendix_panel(ax, panel, title, cmap=cmap or "gray", is_error=is_error)
            if title.startswith("error "):
                add_metric_label(ax, panel_text(row, col))
    for empty_row in range(len(rows), max_rows):
        for col in range(2):
            axes[empty_row, col].axis("off")
    finish_axes(axes)


def make_appendix_dense_page_figure(sample_name, reference, chunk):
    reference_data = chunk[0][1]
    max_rows = max(len(appendix_model_rows(data)) for _, data in chunk)
    fig = plt.figure(figsize=(max(10.0, 2.4 * len(chunk) + 3.0), 7.5), constrained_layout=True)
    fig.suptitle(output_page_title(sample_name, reference["dataset"]), fontsize=9)
    outer = fig.subfigures(
        2,
        1,
        height_ratios=[0.9, 3.4],
        hspace=0.035,
    )
    draw_ground_truth(outer[0], reference_data, sample_name, reference["dataset"])

    flat_slots = list(np.asarray(outer[1].subfigures(1, len(chunk), wspace=0.03), dtype=object).flat)
    for model_index, subfig in enumerate(flat_slots):
        if model_index < len(chunk):
            row, data = chunk[model_index]
            draw_appendix_model_block(subfig, row, data, max_rows)
        else:
            subfig.set_visible(False)
    return fig


def appendix_layout_group(data):
    return "multi-output" if data["architecture"] == "pin" and data.get("pred_mask") is not None else "output-only"


def appendix_prediction_chunks(sample_rows):
    sample_rows = sample_rows.sort_values(["architecture", "model"])
    cached_predictions = [(row, predict_sample(row)) for _, row in sample_rows.iterrows()]
    groups = {"multi-output": [], "output-only": []}
    for item in cached_predictions:
        groups[appendix_layout_group(item[1])].append(item)
    ordered = groups["multi-output"] + groups["output-only"]
    chunks = []
    start = 0
    while start < len(ordered):
        group = appendix_layout_group(ordered[start][1])
        end = start
        while end < len(ordered) and appendix_layout_group(ordered[end][1]) == group:
            end += 1
        group_items = ordered[start:end]
        chunks.extend(
            group_items[i:i + MODELS_PER_OUTPUT_PAGE]
            for i in range(0, len(group_items), MODELS_PER_OUTPUT_PAGE)
        )
        start = end
    return chunks


def preview_appendix_dense_page(appendix=None, sample_name=None, page_index=1):
    if appendix is None:
        _, appendix = build_thesis_plans()
    if appendix.empty:
        print("No appendix rows to preview.")
        return None
    if sample_name is None:
        sample_name = appendix["sample_name"].iloc[0]
    sample_rows = appendix.loc[appendix["sample_name"].astype(str) == str(sample_name)]
    if sample_rows.empty:
        raise ValueError(f"No appendix rows for sample {sample_name!r}.")
    reference = sample_rows.iloc[0]
    chunks = appendix_prediction_chunks(sample_rows)
    if page_index < 1 or page_index > len(chunks):
        raise ValueError(f"page_index must be between 1 and {len(chunks)}.")
    fig = make_appendix_dense_page_figure(sample_name, reference, chunks[page_index - 1])
    display(fig)
    plt.close(fig)
    return fig



def architecture_folder_name(row):
    architecture = str(row.get("architecture", infer_architecture(row)))
    if architecture == "pin":
        return "multi-output"
    if architecture == "oneoutput":
        return "one-output"
    if architecture == "twooutput":
        return "two-output"
    return safe_filename_part(architecture)


def output_variant_folder_name(row):
    model = str(row.get("model", ""))
    if model.startswith("two-output"):
        if "reconstruction" in model:
            return "reconstruction"
        if "Tversky" in model or "tversky" in model:
            return "tversky"
        if "L1" in model or "Dice" in model:
            return "L1-Dice"
    if model.startswith("one-output"):
        if "Tversky" in model or "tversky" in model:
            return "tversky"
        if "Dice" in model or "L1" in model:
            return "Dice-L1"
    if model.startswith("multi-output"):
        if "BCE" in model or "MSE" in model:
            return "BCE-MSE"
        if "Tversky" in model or "tversky" in model:
            return "tversky"
    return safe_filename_part(model)


def appendix_example_spot_folder(root, row):
    folder = root / architecture_folder_name(row) / output_variant_folder_name(row)
    folder.mkdir(parents=True, exist_ok=True)
    return folder

def save_appendix_outputs_by_spot(rows, folder):
    saved = []
    for sample_name, sample_rows in rows.groupby("sample_name", sort=False):
        reference = sample_rows.iloc[0]
        chunks = appendix_prediction_chunks(sample_rows)

        for page_index, chunk in enumerate(chunks, start=1):
            fig = make_appendix_dense_page_figure(sample_name, reference, chunk)
            page_folder = appendix_example_spot_folder(folder, chunk[0][0])
            path = page_folder / (
                f"{safe_filename_part(sample_name)}"
                f"__{safe_filename_part(output_variant_folder_name(chunk[0][0]))}"
                f"__page-{page_index:02d}.pdf"
            )
            fig.savefig(path, format="pdf", dpi=PDF_DPI, bbox_inches="tight", facecolor="white")
            plt.close(fig)
            saved.append({
                "sample_name": sample_name,
                "page": page_index,
                "models_on_page": len(chunk),
                "includes_ground_truth": True,
                "pdf_path": str(path.resolve()),
            })
            print(f"Saved {path}")
    return pd.DataFrame(saved)


def metrics_export_table(rows):
    sample_columns = [
        "section", "role", "role_group", "rank_within_role", "selection", "model", "model_original", "architecture", "dataset", "sample_name",
        "spot_recognition_mean", "dice_mean", "iou_mean", "rmse_mean",
        "spot_0_spot_recognition", "spot_0_dice", "spot_0_iou", "spot_0_rmse", "spot_0_nrmse",
        "spot_1_spot_recognition", "spot_1_dice", "spot_1_iou", "spot_1_rmse", "spot_1_nrmse",
        "model_path", "source_results_csv", "source_file",
    ]
    return rows[existing_columns(rows, sample_columns)].copy()


def export_master_thesis_examples():
    deep, appendix = preview_thesis_plans()
    deep_dir = OUTPUT_ROOT / "in_depth"
    appendix_dir = EXAMPLE_SPOTS_ROOT
    deep_dir.mkdir(parents=True, exist_ok=True)
    appendix_dir.mkdir(parents=True, exist_ok=True)

    deep_records = []
    for _, row in deep.iterrows():
        pdf = save_plan_pdf(row, deep_dir)
        deep_records.append({**row.to_dict(), "pdf_path":str(pdf.resolve())})
        print(f"Saved {pdf}")
    deep_manifest = pd.DataFrame(deep_records)
    deep_manifest.to_csv(deep_dir / "export_manifest.csv", index=False)
    metrics_export_table(deep_manifest).to_csv(deep_dir / "exact_metrics.csv", index=False)
    selected_spot_metrics(deep).to_csv(deep_dir / "exact_spot_metrics.csv", index=False)

    appendix_records = []
    if APPENDIX_SAVE_INDIVIDUAL_PDFS:
        for _, row in appendix.iterrows():
            pdf = save_plan_pdf(row, appendix_dir)
            appendix_records.append({**row.to_dict(), "pdf_path":str(pdf.resolve())})
            print(f"Saved {pdf}")
    if APPENDIX_SAVE_MULTIPAGE_PDFS:
        for sample_name, rows in appendix.groupby("sample_name", sort=False):
            pdf_path = appendix_dir / f"appendix__{safe_filename_part(sample_name)}__all-models.pdf"
            with PdfPages(pdf_path) as pdf:
                for _, row in rows.iterrows():
                    fig = plan_figure(row)
                    pdf.savefig(fig, dpi=PDF_DPI, bbox_inches="tight", facecolor="white")
                    plt.close(fig)
            print(f"Saved {pdf_path}")
    if APPENDIX_SAVE_COMBINED_BY_SPOT_PDFS and not appendix.empty:
        combined_manifest = save_appendix_outputs_by_spot(appendix, appendix_dir)
        combined_manifest.to_csv(appendix_dir / "combined_by_spot_manifest.csv", index=False)
    appendix_manifest = pd.DataFrame(appendix_records) if appendix_records else appendix
    appendix_manifest.to_csv(appendix_dir / "export_manifest.csv", index=False)
    metrics_export_table(appendix_manifest).to_csv(appendix_dir / "exact_metrics.csv", index=False)
    selected_spot_metrics(appendix).to_csv(appendix_dir / "exact_spot_metrics.csv", index=False)

    if not best_worst_spot_table.empty:
        best_worst_spot_table.to_csv(OUTPUT_ROOT / "best_worst_spots_per_model.csv", index=False)

    print(f"\nIn-depth: {deep_dir.resolve()}\nAppendix: {appendix_dir.resolve()}")


def export_selected_model_outputs():
    return export_master_thesis_examples()


def export_selected_model_outpus():
    return export_selected_model_outputs()




## Real figure preview

Run this after the helper functions are loaded to render one in-depth figure and one dense appendix page.

In [ ]:
preview_thesis_figures(deep_plan, appendix_plan)


## Export

Run the next cell when the preview looks right.

In [ ]:
export_master_thesis_examples()
